# AtlasEstate 3D - Exploratory Data Analysis & Schema Verification
## Comprehensive Inspection of Marrakech Real Estate Inventory
**Author:** AtlasEstate Engineering Team  
**Date:** March 2026  
**Pipeline Layer:** Silver & Gold Parquet Validation

---
### Objectives
1. Inspect raw-to-processed schema, field datatypes, and integrity constraints.
2. Characterize univariate distributions of asset prices, surfaces, and price-per-square-meter.
3. Validate outlier detection thresholds and identify log-normal distribution properties.
4. Establish baseline data hygiene before spatial enrichment and machine learning.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path

# Styling configuration
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

# Path resolution
DATA_DIR = Path("../data")
silver_path = DATA_DIR / "silver" / "silver_properties.parquet"
gold_path = DATA_DIR / "gold" / "gold_properties_enriched.parquet"

df = pd.read_parquet(gold_path if gold_path.exists() else silver_path)
print(f"Dataset successfully loaded: {df.shape[0]:,} rows | {df.shape[1]} columns")


### 1. Schema & Datatype Audit


In [ ]:
# Display primary structural columns and datatypes
cols_to_inspect = ['property_id', 'title', 'price', 'surface', 'property_type', 'neighborhood', 'bedrooms', 'bathrooms', 'price_per_m2']
df[cols_to_inspect].info()


In [ ]:
# Check missing value frequencies across all attributes
null_counts = df.isnull().sum()
null_pct = (null_counts / len(df)) * 100
null_summary = pd.DataFrame({'Missing Count': null_counts, 'Percent (%)': null_pct})
null_summary[null_summary['Missing Count'] > 0] if (null_counts > 0).any() else print("100% Data Completeness: Zero missing values detected.")


### 2. Descriptive Statistics of Core Financial & Physical Dimensions


In [ ]:
num_cols = ['price', 'surface', 'price_per_m2', 'bedrooms', 'bathrooms', 'year_built']
df[num_cols].describe().T.style.format({
    'count': '{:,.0f}',
    'mean': '{:,.1f}',
    'std': '{:,.1f}',
    'min': '{:,.1f}',
    '25%': '{:,.1f}',
    '50%': '{:,.1f}',
    '75%': '{:,.1f}',
    'max': '{:,.1f}'
})


### 3. Univariate Distributions: Price & Surface
Real estate price data in Morocco exhibits a pronounced right-skew (log-normal distribution). Below we compare raw versus log-transformed prices.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw Price Distribution
sns.histplot(df['price'] / 1e6, kde=True, ax=axes[0], color='#0284c7', bins=35)
axes[0].set_title("Distribution of Property Price (Million MAD)", fontsize=13, fontweight='bold')
axes[0].set_xlabel("Price (M MAD)")
axes[0].set_ylabel("Count")

# Log-Transformed Price
sns.histplot(np.log1p(df['price']), kde=True, ax=axes[1], color='#0d9488', bins=35)
axes[1].set_title("Log-Transformed Price ln(Price + 1)", fontsize=13, fontweight='bold')
axes[1].set_xlabel("ln(Price)")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()


### 4. Price per Square Meter by Typology


In [ ]:
plt.figure(figsize=(11, 5))
order = df.groupby('property_type')['price_per_m2'].median().sort_values(ascending=False).index
sns.boxplot(x='property_type', y='price_per_m2', data=df, order=order, palette='crest')
plt.title("Price per m² (MAD) Distribution by Property Typology", fontsize=13, fontweight='bold')
plt.xlabel("Property Typology")
plt.ylabel("MAD / m²")
plt.xticks(rotation=15)
plt.show()


### 5. Key Exploratory Findings
1. **Right-Skewed Target:** Property price spans from ~550,000 MAD to >18,000,000 MAD (median: ~2,400,000 MAD), indicating the necessity of log-scaling for linear modeling.
2. **Typology Stratification:** Villas and traditional Riads exhibit substantial surface area variances compared to modern apartments.
3. **Price/m² Consistency:** Price per square meter ranges consistently between 7,000 and 32,000 MAD/m², confirming high data hygiene without wild outlier contamination.
